<a href="https://colab.research.google.com/github/muhammadabdurrehmanmaqsood/brain-tumor-classifier/blob/main/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri -p /content/drive/MyDrive/brain_tumor_project/data

Dataset URL: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
License(s): MIT
100% 86.8M/86.8M [00:00<00:00, 138MB/s]



In [6]:
import os
from google.colab import drive

# 1. Mount Google Drive safely
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted!")

# 2. Extract the dataset from Drive to Colab's fast local storage
!unzip -q -o /content/drive/MyDrive/brain_tumor_project/data/brain-tumor-classification-mri.zip -d /content/dataset/

# 3. Import PyTorch and build the DataLoaders
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Define augmentations
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets from the extracted folders
full_train_dataset = datasets.ImageFolder(root='/content/dataset/Training', transform=train_transforms)

# Create the Validation Split (85% Train, 15% Val)
train_size = int(0.85 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

generator = torch.Generator().manual_seed(42) # Ensures the split is consistent
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=generator)

# Override the transform for the validation subset so it doesn't get random flips/rotations
val_dataset.dataset.transform = val_transforms

# Package into DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Data ready. Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

Drive already mounted!
Data ready. Training batches: 77, Validation batches: 14


In [7]:
import torch
import torch.nn as nn
from torchvision import models

# 1. Hardware configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load pretrained ResNet50
# Weights parameter replaces the old 'pretrained=True' argument
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# 3. Replace the final Fully Connected (fc) layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4) # 4 represents our four classes

# 4. Move the entire model to the GPU
model = model.to(device)

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]


In [8]:
import torch.optim as optim

# CrossEntropyLoss is standard for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer with a small learning rate for fine-tuning
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Optional: Number of times to loop over the entire dataset
EPOCHS = 10

In [9]:
import os

# Ensure the models directory exists in your Google Drive
save_dir = '/content/drive/MyDrive/brain_tumor_project/models'
os.makedirs(save_dir, exist_ok=True)
best_filepath = os.path.join(save_dir, 'resnet50_best.pth')

best_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print("-" * 10)

    # --- TRAINING PHASE ---
    model.train() # Set model to training mode
    running_loss = 0.0
    running_corrects = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = running_corrects.double() / len(train_dataset)
    print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

    # --- VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode (disables dropout, etc.)
    val_loss = 0.0
    val_corrects = 0

    # Disable gradient calculation for validation (saves memory/compute)
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += torch.sum(preds == labels.data)

    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = val_corrects.double() / len(val_dataset)
    print(f"Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}")

    # --- MLOPS CHECKPOINTING ---
    # Save the model only if it improved on the validation set
    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        torch.save(model.state_dict(), best_filepath)
        print(f"*** New best model saved to Drive with accuracy: {best_val_acc:.4f} ***")

    print("\n")

Epoch 1/10
----------
Train Loss: 0.5765 Acc: 0.8073
Val Loss: 0.1614 Acc: 0.9397
*** New best model saved to Drive with accuracy: 0.9397 ***


Epoch 2/10
----------
Train Loss: 0.0944 Acc: 0.9709
Val Loss: 0.1245 Acc: 0.9536
*** New best model saved to Drive with accuracy: 0.9536 ***


Epoch 3/10
----------
Train Loss: 0.0432 Acc: 0.9865
Val Loss: 0.1446 Acc: 0.9466


Epoch 4/10
----------
Train Loss: 0.0263 Acc: 0.9934
Val Loss: 0.1224 Acc: 0.9606
*** New best model saved to Drive with accuracy: 0.9606 ***


Epoch 5/10
----------
Train Loss: 0.0202 Acc: 0.9947
Val Loss: 0.1510 Acc: 0.9420


Epoch 6/10
----------
Train Loss: 0.0223 Acc: 0.9930
Val Loss: 0.1164 Acc: 0.9582


Epoch 7/10
----------
Train Loss: 0.0291 Acc: 0.9926
Val Loss: 0.1660 Acc: 0.9490


Epoch 8/10
----------
Train Loss: 0.0264 Acc: 0.9926
Val Loss: 0.1089 Acc: 0.9582


Epoch 9/10
----------
Train Loss: 0.0117 Acc: 0.9971
Val Loss: 0.0857 Acc: 0.9745
*** New best model saved to Drive with accuracy: 0.9745 ***


Epoc